In [ ]:
import pandas as pd
import numpy as np

# scikit‑learn imports
from sklearn.model_selection   import train_test_split
from sklearn.preprocessing     import StandardScaler, LabelEncoder
from sklearn.neighbors         import KNeighborsClassifier
from sklearn.cluster           import KMeans
from sklearn.metrics           import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    silhouette_score
)

# 1. Load your data
df = pd.read_csv('email_phishing_data.csv', low_memory=False)

# 2. Split off the target
#    — here we assume the last column is your label (e.g. “phishing” vs. “legit”)
target_col = df.columns[-1]
X_raw       = df.drop(columns=[target_col])
y_raw       = df[target_col]

# 3. Preprocess features
#    • Keep only numeric features (you can also one‑hot encode or hash high‑cardinality categoricals)
#    • Drop any columns that aren’t useful, handle missing values, etc.
X_num = X_raw.select_dtypes(include=[np.number]).dropna(axis=1)

# 4. Encode the label if it’s text
if y_raw.dtype == 'object':
    lbl = LabelEncoder()
    y = lbl.fit_transform(y_raw)
else:
    y = y_raw.values

# 5. — SUPERVISED: K‑NEAREST NEIGHBORS CLASSIFIER —
#    a) train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_num, y, test_size=0.3, random_state=0
)

#    b) scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

#    c) fit & predict
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train)
y_pred = knn.predict(X_test_s)

#    d) evaluate
print("KNN Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


# 6. — UNSUPERVISED: K‑MEANS CLUSTERING —
#    a) apply to your full (scaled) numeric dataset
X_full_s = scaler.fit_transform(X_num)

kmeans = KMeans(n_clusters=2, random_state=0)
clusters = kmeans.fit_predict(X_full_s)

#    b) evaluate clustering
print("\nCluster Counts:", np.bincount(clusters))
print("Silhouette Score:", silhouette_score(X_full_s, clusters))


# 7. Next steps / tuning ideas
#    • For KNN: try different k, weight schemes, distance metrics
#    • For KMeans: experiment with n_clusters > 2, mini‑batch KMeans, or PCA beforehand
#    • Feature engineering: you’ll likely want to transform text fields (e.g. TF‑IDF), dates, or categorical flags
